# Lifetime Model

**Author:** DLHub

**Date:** 09/03/2026

# 0. Instalación de paquetes en ADA

**Nota:** Dado que no contamos con GPU's o PyTorch Distribuido en el código proceso los datos con `PyArrow` y `Polars` para que procesarlos en memoria de la instancia. Esto tiene implicaciones en el performance del código, pero confío en que se podrá resolver más adelante.

In [ ]:
%pip install torch torchvision matplotlib -q

In [ ]:
%pip install scikit-learn pandas polars awswrangler s3fs pyarrow -q

# 1. Libraries

In [ ]:
import os
import hashlib
from dataclasses import dataclass
from typing import Optional, Iterable

import numpy as np
import polars as pl
import pyarrow.dataset as ds
import pyarrow.fs as pafs
import torch
import torch.nn as nn
from torch.utils.data import IterableDataset, DataLoader, get_worker_info

# 2. Config & Constants

# 2.1 Explorando Datos para train y test

In [ ]:
TRAIN_PATH = "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_entrena_target_independiente"

In [ ]:
TEST_PATH = "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_valida_target_independiente"

In [ ]:
%%bash
aws s3 ls s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_entrena_target_independiente --recursive | awk -F"/" '{print $4}' | sort | uniq -c

El comando previo arroja las siguientes particiones
```
1 _SUCCESS
    117 cutoff_date=2024-11-01
    266 cutoff_date=2024-12-01
    368 cutoff_date=2025-01-01
    414 cutoff_date=2025-02-01
    415 cutoff_date=2025-03-01
    386 cutoff_date=2025-04-01
    268 cutoff_date=2025-05-01
    250 cutoff_date=2025-06-01
    350 cutoff_date=2025-07-01
    255 cutoff_date=2025-08-01
```

In [ ]:
%%bash
aws s3 ls s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_valida_target_independiente --recursive | awk -F"/" '{print $4}' | sort | uniq -c

Estas son las particiones en la segunda ruta:
```
      1 _SUCCESS
    190 cutoff_date=2025-09-01
    112 cutoff_date=2025-10-01
    313 cutoff_date=2025-11-01
    306 cutoff_date=2025-12-01
```

## 2.2 Paths que se usarán train y test

In [ ]:
# Solo vamos a entrenar con la ultima partición de s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_entrena_target_independiente/
PATHS = [
    #"s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_entrena_target_independiente/cutoff_date=2025-06-01/",
    #"s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_entrena_target_independiente/cutoff_date=2025-07-01/",
    "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_entrena_target_independiente/cutoff_date=2025-08-01/",
]


# Aqui consideramos todas las particiones
VALID_PATHS = [
    "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_valida_target_independiente/cutoff_date=2025-09-01/",
    "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_valida_target_independiente/cutoff_date=2025-10-01/",
    "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_valida_target_independiente/cutoff_date=2025-11-01/",
    "s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_valida_target_independiente/cutoff_date=2025-12-01/",
]

## 2.3 Configuraciones para el modelo

In [ ]:
TARGET_COL = "independent"
ID_COL = "customer_id"

NUMERIC_COLS = [
    "transct_size",
    "product_size",
    "independent_unmes",
    "avg_transactions_amount_imputed",
    "bhsc_seg_score",
    "total_transaction_number_imputed",
    "cpac_average_balance_amount_med",
    "account_esntl_expenses_amount_imputed",
    "non_credit_incm_total_amount_imputed",
    "avg_essential_ticket",
    "account_nesl_expenses_amount_imputed",
    "seniority",
    "token_score",
    "essential_expense_ratio",
    "income_score",
    "age_clean",
    "month_cpac_srty_max_number_imputed",
    "non_essential_ratio",
    "bhsc_mdl_cust_tot_scor_number_med",
    "segment_score",
    "incm_uniq_model_result_number_med",
    "foodie_unmes",
    "avg_captation_balance_amount_med",
    "title_score",
    "young_and_single",
    "os_score",
    "seguro_salud",
    "student_unmes",
    "seguro_salud_viaje",
    "is_married_like",
]

AWS_REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")

HASH_BUCKET_SIZE = 2**18
EMBEDDING_DIM = 16

BATCH_SIZE_ROWS = 50_000
TRAIN_BATCH_SIZE = 1024
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

EPOCHS = 3
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

FILL_NULL_VALUE = 0.0
SHUFFLE_WITHIN_BATCH = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 12345

# 3. Auxiliar Functions
## 3.1 Utilities

In [ ]:
def set_seed(seed: int) -> None:
    """
    Configura la semilla de aleatoriedad para NumPy y PyTorch.

    Establece un estado determinista en las librerías de generación de números 
    aleatorios, incluyendo el soporte para GPU (CUDA) si está disponible.

    Args:
        seed (int): El valor numérico de la semilla a utilizar.
    """
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def stable_hash_bucket(value: str, num_buckets: int) -> int:
    """
        Asigna una cadena de texto a un contenedor (bucket) de forma determinista.

        Utiliza el algoritmo MD5 para garantizar que un mismo valor siempre caiga 
        en el mismo bucket, incluso entre diferentes sesiones de Python.

        Args:
            value (str): La cadena de texto a procesar. Si es None, se usa un valor por defecto.
            num_buckets (int): El número total de contenedores disponibles.

        Returns:
            int: El índice del bucket asignado, calculado como $hash \pmod n$.
    """
    if value is None:
        value = "__MISSING__"
    digest = hashlib.md5(str(value).encode("utf-8")).hexdigest()
    return int(digest, 16) % num_buckets


def get_s3_filesystem(region: str = AWS_REGION) -> pafs.S3FileSystem:
    """
        Crea una instancia del sistema de archivos S3 de PyArrow.

        Args:
            region (str, opcional): La región de AWS donde se encuentra el bucket. 
                Por defecto utiliza `AWS_REGION`.

        Returns:
            pafs.S3FileSystem: Un objeto de conexión listo para interactuar con S3.
    """
    return pafs.S3FileSystem(region=region)


def extract_worker_paths(paths: list[str]) -> list[str]:
    """
        Divide una lista de rutas entre los diferentes trabajadores (workers) activos.

        Utiliza una técnica de 'striding' (salto) basada en el ID del trabajador 
        para asegurar que cada proceso reciba un subconjunto único de archivos, 
        evitando duplicación de datos en procesos paralelos.

        Args:
            paths (list[str]): Lista completa de rutas de archivos a procesar.

        Returns:
            list[str]: Subconjunto de rutas asignadas específicamente al trabajador actual. 
                Si no hay información de workers, devuelve la lista original.
    """
    worker_info = get_worker_info()
    if worker_info is None:
        return paths
    return paths[worker_info.id::worker_info.num_workers]


def parse_s3_uri(uri: str) -> tuple[str, str]:
    """
    Convierte:
        s3://bucket/key...
    en:
        ("bucket", "key...")
    """
    if not uri.startswith("s3://"):
        raise ValueError(f"Se esperaba URI s3://..., recibido: {uri}")
    no_scheme = uri[5:]
    parts = no_scheme.split("/", 1)
    bucket = parts[0]
    key = parts[1] if len(parts) > 1 else ""
    return bucket, key


def normalize_s3_path(path: str) -> str:
    """
    Convierte:
      - s3://bucket/key
      - bucket/key
    a formato PyArrow S3FileSystem:
      - bucket/key
    """
    if path.startswith("s3://"):
        bucket, key = parse_s3_uri(path)
        return f"{bucket}/{key}".rstrip("/")
    return path.rstrip("/")


def expand_s3_paths(paths: list[str], region: str = AWS_REGION) -> list[str]:
    """
    Expande prefijos S3 a archivos parquet reales.
    Admite:
      - s3://bucket/prefix/
      - s3://bucket/file.parquet
      - bucket/prefix/
      - bucket/file.parquet

    Devuelve una lista de rutas tipo:
      bucket/key/file.parquet
    """
    fs = get_s3_filesystem(region=region)
    out = []

    for raw_path in paths:
        path = normalize_s3_path(raw_path)

        if "*" in path:
            raise ValueError(
                f"No uses globs en PATHS con PyArrow S3FileSystem: {raw_path}\n"
                f"Usa el prefijo del directorio, por ejemplo .../cutoff_date=2025-08-01/"
            )

        if path.endswith(".parquet"):
            info = fs.get_file_info([path])[0]
            if info.type == pafs.FileType.File:
                out.append(path)
            else:
                raise FileNotFoundError(f"No existe el archivo parquet: {raw_path}")
            continue

        selector = pafs.FileSelector(path, recursive=True)
        infos = fs.get_file_info(selector)

        parquet_files = [
            info.path
            for info in infos
            if info.type == pafs.FileType.File and info.path.endswith(".parquet")
        ]

        if not parquet_files:
            raise FileNotFoundError(f"No se encontraron .parquet bajo el prefijo: {raw_path}")

        out.extend(sorted(parquet_files))

    if not out:
        raise ValueError("No se resolvió ningún archivo parquet.")

    return out

# 3.2 Streaming & others

In [ ]:
class StreamingStandardizer:
    """
        Calculador de media y desviación estándar de forma incremental.

        Mantiene sumas acumuladas de los datos para permitir el cálculo de 
        estadísticas de normalización sin necesidad de cargar todo el 
        dataset en memoria simultáneamente.

        Attributes:
            n (int): Número total de muestras procesadas.
            sum (np.ndarray): Suma acumulada de las características.
            sum_sq (np.ndarray): Suma acumulada de los cuadrados de las características.
    """
    def __init__(self, n_features: int):
        self.n = 0
        self.sum = np.zeros(n_features, dtype=np.float64)
        self.sum_sq = np.zeros(n_features, dtype=np.float64)

    def update(self, x: np.ndarray) -> None:
        if x.size == 0:
            return
        self.n += x.shape[0]
        self.sum += x.sum(axis=0)
        self.sum_sq += np.square(x).sum(axis=0)

    def finalize(self) -> tuple[np.ndarray, np.ndarray]:
        if self.n == 0:
            raise ValueError("No se encontraron filas para calcular mean/std.")
        mean = self.sum / self.n
        var = self.sum_sq / self.n - np.square(mean)
        var = np.maximum(var, 1e-12)
        std = np.sqrt(var)
        return mean.astype(np.float32), std.astype(np.float32)


def iter_arrow_batches(
    paths: list[str],
    columns: list[str],
    batch_size_rows: int,
    region: str = AWS_REGION,
) -> Iterable[pl.DataFrame]:
    """
    Generador que lee archivos Parquet de S3 en lotes (batches) de Polars.

    Args:
        paths (list[str]): Lista de rutas o prefijos de S3.
        columns (list[str]): Nombres de las columnas a cargar.
        batch_size_rows (int): Tamaño máximo de cada lote en número de filas.
        region (str): Región de AWS donde residen los datos.

    Yields:
        Iterable[pl.DataFrame]: Lotes de datos convertidos a DataFrames de Polars.
    """
    filesystem = get_s3_filesystem(region=region)
    resolved_files = expand_s3_paths(paths, region=region)

    dataset = ds.dataset(
        resolved_files,
        format="parquet",
        filesystem=filesystem,
    )

    scanner = dataset.scanner(
        columns=columns,
        batch_size=batch_size_rows,
        use_threads=True,
    )

    for record_batch in scanner.to_batches():
        yield pl.from_arrow(record_batch)


def compute_streaming_stats(
    paths: list[str],
    numeric_cols: list[str],
    batch_size_rows: int = BATCH_SIZE_ROWS,
    fill_null_value: float = FILL_NULL_VALUE,
    region: str = AWS_REGION,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Calcula la media y desviación estándar de columnas numéricas en un dataset de S3.

    Procesa los datos de forma iterativa para evitar desbordamientos de memoria,
    manejando valores nulos y realizando conversiones de tipo sobre la marcha.

    Args:
        paths (list[str]): Rutas de los archivos Parquet.
        numeric_cols (list[str]): Lista de columnas numéricas a procesar.
        batch_size_rows (int, opcional): Cantidad de filas por lote.
        fill_null_value (float, opcional): Valor para imputar nulos.
        region (str, opcional): Región de AWS.

    Returns:
        tuple[np.ndarray, np.ndarray]: Vectores de (media, desviación_estándar) 
            de longitud igual a `len(numeric_cols)`.
    """
    stats = StreamingStandardizer(len(numeric_cols))

    for batch_df in iter_arrow_batches(
        paths=paths,
        columns=numeric_cols,
        batch_size_rows=batch_size_rows,
        region=region,
    ):
        batch_df = batch_df.with_columns([
            pl.col(c).cast(pl.Float32).fill_null(fill_null_value).alias(c)
            for c in numeric_cols
        ])

        x = batch_df.select(numeric_cols).to_numpy().astype(np.float32, copy=False)
        stats.update(x)

    return stats.finalize()

In [ ]:
class S3ParquetArrowIterableDataset(IterableDataset):
    """
    Dataset iterable para cargar archivos Parquet desde S3 directamente a PyTorch.

    Esta clase implementa una carga de datos mediante streaming, evitando
    descargar archivos completos a disco. Soporta normalización (scaling),
    imputación de nulos, hashing de IDs y distribución automática entre workers
    de PyTorch para entrenamiento en paralelo.

    Attributes:
        paths (list[str]): Rutas o prefijos de S3 de los archivos Parquet.
        target_col (str): Nombre de la columna objetivo (label).
        numeric_cols (list[str]): Lista de columnas numéricas (features).
        id_col (Optional[str]): Columna identificadora para hashing categórico.
        batch_size_rows (int): Tamaño del lote cargado desde el scanner de Arrow.
        hash_bucket_size (int): Tamaño del espacio de buckets para el hashing del ID.
        fill_null_value (float): Valor para rellenar nulos en columnas numéricas.
        scaler_mean (Optional[np.ndarray]): Media para normalización estándar.
        scaler_std (Optional[np.ndarray]): Desviación para normalización estándar.
        shuffle_within_batch (bool): Si es True, baraja las filas dentro de cada lote.
        region (str): Región de AWS para la conexión de S3.
    """

    def __init__(
        self,
        paths: list[str],
        target_col: str,
        numeric_cols: list[str],
        id_col: Optional[str] = None,
        batch_size_rows: int = BATCH_SIZE_ROWS,
        hash_bucket_size: int = HASH_BUCKET_SIZE,
        fill_null_value: float = FILL_NULL_VALUE,
        scaler_mean: Optional[np.ndarray] = None,
        scaler_std: Optional[np.ndarray] = None,
        shuffle_within_batch: bool = False,
        region: str = AWS_REGION,
    ):
        super().__init__()
        self.paths = paths
        self.target_col = target_col
        self.numeric_cols = numeric_cols
        self.id_col = id_col
        self.batch_size_rows = batch_size_rows
        self.hash_bucket_size = hash_bucket_size
        self.fill_null_value = fill_null_value
        self.scaler_mean = None if scaler_mean is None else np.asarray(scaler_mean, dtype=np.float32)
        self.scaler_std = None if scaler_std is None else np.asarray(scaler_std, dtype=np.float32)
        self.shuffle_within_batch = shuffle_within_batch
        self.region = region

        self.required_cols = list(numeric_cols) + [target_col]
        if id_col is not None:
            self.required_cols.append(id_col)

    def __iter__(self):
        """
        Itera sobre los datos realizando transformaciones en tiempo real.

        El proceso sigue este flujo:
        1. Filtra las rutas según el worker actual de PyTorch.
        2. Abre el dataset en S3 usando PyArrow.
        3. Convierte batches de Arrow a DataFrames de Polars.
        4. Realiza cast de tipos e imputación de valores nulos.
        5. Aplica normalización estándar: $z = \frac{x - \mu}{\sigma}$.
        6. Calcula hashes estables para la columna de ID si existe.

        Yields:
            tuple: Dependiendo de la configuración, devuelve:
                - (x_num, y) si id_col es None.
                - (x_num, x_id, y) si id_col está definido.
        """

        worker_paths = extract_worker_paths(self.paths)
        if not worker_paths:
            return

        filesystem = get_s3_filesystem(region=self.region)
        resolved_files = expand_s3_paths(worker_paths, region=self.region)

        dataset = ds.dataset(
            resolved_files,
            format="parquet",
            filesystem=filesystem,
        )

        scanner = dataset.scanner(
            columns=self.required_cols,
            batch_size=self.batch_size_rows,
            use_threads=True,
        )

        for record_batch in scanner.to_batches():
            batch_df = pl.from_arrow(record_batch)

            exprs = [
                pl.col(c).cast(pl.Float32).fill_null(self.fill_null_value).alias(c)
                for c in self.numeric_cols
            ]
            exprs.append(
                pl.col(self.target_col).cast(pl.Float32).fill_null(0.0).alias(self.target_col)
            )

            if self.id_col is not None:
                exprs.append(
                    pl.col(self.id_col).cast(pl.Utf8).fill_null("__MISSING__").alias(self.id_col)
                )

            batch_df = batch_df.with_columns(exprs)

            x_num = batch_df.select(self.numeric_cols).to_numpy().astype(np.float32, copy=False)
            y = batch_df.get_column(self.target_col).to_numpy().astype(np.float32, copy=False)

            if self.scaler_mean is not None and self.scaler_std is not None:
                x_num = (x_num - self.scaler_mean) / np.clip(self.scaler_std, 1e-8, None)

            if self.id_col is not None:
                id_values = batch_df.get_column(self.id_col).to_list()
                x_id = np.array(
                    [stable_hash_bucket(v, self.hash_bucket_size) for v in id_values],
                    dtype=np.int64,
                )
            else:
                x_id = None

            indices = np.arange(len(batch_df))
            if self.shuffle_within_batch:
                np.random.shuffle(indices)

            for i in indices:
                sample = {
                    "x_num": torch.from_numpy(x_num[i]),
                    "y": torch.tensor(y[i], dtype=torch.float32),
                }

                if x_id is not None:
                    sample["x_id"] = torch.tensor(x_id[i], dtype=torch.long)

                yield sample

In [ ]:
def collate_tabular(batch: list[dict]) -> dict[str, torch.Tensor]:
    x_num = torch.stack([item["x_num"] for item in batch], dim=0)
    y = torch.stack([item["y"] for item in batch], dim=0)

    output = {"x_num": x_num, "y": y}
    if "x_id" in batch[0]:
        output["x_id"] = torch.stack([item["x_id"] for item in batch], dim=0)
    return output

In [ ]:
def compute_class_balance(
    paths: list[str],
    target_col: str,
    batch_size_rows: int = BATCH_SIZE_ROWS,
    region: str = AWS_REGION,
) -> tuple[int, int]:
    """
    Devuelve:
        n_neg, n_pos
    calculados en streaming sobre train.
    """
    n_pos = 0
    n_neg = 0

    for batch_df in iter_arrow_batches(
        paths=paths,
        columns=[target_col],
        batch_size_rows=batch_size_rows,
        region=region,
    ):
        y = (
            batch_df
            .with_columns(pl.col(target_col).cast(pl.Int64).fill_null(0).alias(target_col))
            .get_column(target_col)
            .to_numpy()
        )

        n_pos += int((y == 1).sum())
        n_neg += int((y == 0).sum())

    return n_neg, n_pos

# 4. Model

Nota: En el modelo se usa un embedding que represente el `customer_id` usando una función hashing, dado que dicha variable categórica puede tener cardinalidad alta en producción (véase https://medium.com/nextgenllm/feature-hashing-efficient-categorical-data-encoding-for-large-scale-ml-systems-3d143a8ae239)

In [ ]:
class TabularNN(nn.Module):
    """
    Red Neuronal para datos tabulares con soporte para embeddings del customer_id.

    Esta arquitectura procesa simultáneamente variables numéricas y una variable
    categórica de alta cardinalidad (customer_id) mediante una capa de embedding.
    Ambas representaciones se concatenan y pasan a través de un Perceptrón
    Multicapa (MLP) con bloques de Linear-ReLU-BatchNorm-Dropout.

    Attributes:
        embedding (nn.Embedding): Capa que transforma los índices de hash en
            vectores densos (embeddings).
        mlp (nn.Sequential): Secuencia de capas densas y de regularización que
            realizan la regresión o clasificación.
    """

    def __init__(
        self,
        num_numeric_features: int,
        num_hash_buckets: int,
        emb_dim: int = EMBEDDING_DIM,
        hidden_dims: list[int] = [128, 64],
        dropout: float = 0.2,
    ):
        """
        Inicializa la arquitectura de la red.

        Args:
            num_numeric_features (int): Cantidad de columnas numéricas de entrada.
            num_hash_buckets (int): Tamaño del vocabulario para el embedding 
                (debe coincidir con el `hash_bucket_size` del dataset).
            emb_dim (int, opcional): Dimensión del vector denso para el ID.
            hidden_dims (list[int], opcional): Lista con la cantidad de neuronas 
                por capa oculta.
            dropout (float, opcional): Probabilidad de desactivación de neuronas 
                para regularización.
        """
        super().__init__()

        # Representación densa para customer_id
        self.embedding = nn.Embedding(num_hash_buckets, emb_dim)

        layers = []
        input_dim = num_numeric_features + emb_dim
        prev_dim = input_dim

        for h in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h),
                nn.ReLU(),
                nn.BatchNorm1d(h),
                nn.Dropout(dropout),
            ])
            prev_dim = h

        layers.append(nn.Linear(prev_dim, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x_num: torch.Tensor, x_id: torch.Tensor) -> torch.Tensor:
        """
        Realiza el pase hacia adelante (forward pass) del modelo.

        El flujo de datos es:
        1. $x_{id} \rightarrow \text{Embedding} \rightarrow \text{emb}$
        2. $\text{input} = [x_{num} ; \text{emb}]$ (concatenación)
        3. $\text{input} \rightarrow \text{MLP} \rightarrow \text{output}$

        Args:
            x_num (torch.Tensor): Tensores numéricos de forma $(N, \text{num\_numeric\_features})$.
            x_id (torch.Tensor): Tensores de índices (hashes) de forma $(N,)$.

        Returns:
            torch.Tensor: Predicción escalar para cada muestra, de forma $(N,)$.
        """
        emb = self.embedding(x_id)
        x = torch.cat([x_num, emb], dim=1)
        return self.mlp(x).squeeze(1)

In [ ]:
@dataclass
class TrainArtifacts:
    model: nn.Module
    scaler_mean: np.ndarray
    scaler_std: np.ndarray

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: str,
) -> dict[str, float]:
    """
    Ejecuta un ciclo completo de entrenamiento (época) sobre el dataset.

    La función pone el modelo en modo entrenamiento, itera sobre los lotes del 
    DataLoader, realiza la pasada hacia adelante, calcula la pérdida, ejecuta 
    la retropropagación y actualiza los pesos del optimizador.

    Args:
        model (nn.Module): El modelo de red neuronal a entrenar.
        loader (DataLoader): El cargador de datos que entrega los lotes. 
            Se espera que cada lote sea un diccionario con las llaves 
            "x_num", "x_id" y "y".
        optimizer (torch.optim.Optimizer): El algoritmo de optimización (ej. Adam, SGD).
        criterion (nn.Module): La función de pérdida (ej. BCELoss).
        device (str): El dispositivo donde se realizarán los cálculos ('cuda' o 'cpu').

    Returns:
        dict[str, float]: Un diccionario con las métricas promedio de la época:
            - "loss": Pérdida total promediada por muestra.
            - "accuracy": Precisión (exactitud) del modelo en el rango [0, 1].

    Note:
        Se utiliza `non_blocking=True` en la transferencia de tensores al dispositivo 
        para solapar el copiado de memoria con el cálculo en GPU, siempre que los 
        tensores estén en memoria anclada (pinned memory).
    """
    model.train()

    total_loss = 0.0
    total_examples = 0
    total_correct = 0

    for batch in loader:
        x_num = batch["x_num"].to(device, non_blocking=True)
        x_id = batch["x_id"].to(device, non_blocking=True)
        y = batch["y"].to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(x_num, x_id)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            preds = (torch.sigmoid(logits) >= 0.5).float()
            correct = (preds == y).sum().item()

        bs = y.size(0)
        total_loss += loss.item() * bs
        total_examples += bs
        total_correct += correct

    return {
        "loss": total_loss / max(total_examples, 1),
        "accuracy": total_correct / max(total_examples, 1),
    }

# 5. Traning  & evaluation
# 5.1 Traning Model

Para entrenar el modelo seguimos estos pasos:

* A. Buscamos la data de las particions de la lista `PATHS` y las procesamos con PyArrow + Polars para irlas leyendo en [Streaming](https://docs.pola.rs/user-guide/concepts/streaming/)
* B. Calculamos las media y desviación estándar de las variables numéricas seleccionadas en el pipeline, las cuales se usan para normalizar los datos.
* C. Usando Polars con proceso de iteración sobre la data de entrenamiento, ésta se lee por batches y se normaliza.
* D. Con tales datos se construye un DataLoader de PyTorch
* E. Se hace el settings del resto de objetos para entrenar el modelo, junto con su optimizador y función de pérdida:
    - Especialmente, se emplea `BCEWithLogitsLoss` con el parámetro `pos_weight` para intentar corregir el desbalance de las clases.

In [ ]:
set_seed(SEED) # Establece una semilla

print("Resolviendo archivos parquet...")
resolved = expand_s3_paths(PATHS, region=AWS_REGION)
print(f"Archivos encontrados: {len(resolved)}")

print("Device:", DEVICE)
print("Calculando estadísticas streaming...")
scaler_mean, scaler_std = compute_streaming_stats(
        paths=PATHS,
        numeric_cols=NUMERIC_COLS,
        batch_size_rows=BATCH_SIZE_ROWS,
        fill_null_value=FILL_NULL_VALUE,
        region=AWS_REGION,
    )

print("Construyendo dataset...")
train_ds = S3ParquetArrowIterableDataset(
    paths=PATHS,
    target_col=TARGET_COL,
    numeric_cols=NUMERIC_COLS,
    id_col=ID_COL,
    batch_size_rows=BATCH_SIZE_ROWS,
    hash_bucket_size=HASH_BUCKET_SIZE,
    fill_null_value=FILL_NULL_VALUE,
    scaler_mean=scaler_mean,
    scaler_std=scaler_std,
    shuffle_within_batch=SHUFFLE_WITHIN_BATCH,
    region=AWS_REGION,
)

train_loader = DataLoader(
    train_ds,
    batch_size=TRAIN_BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_tabular,
)

print("Construyendo modelo...")
model = TabularNN(
    num_numeric_features=len(NUMERIC_COLS),
    num_hash_buckets=HASH_BUCKET_SIZE,
    emb_dim=EMBEDDING_DIM,
    hidden_dims=[128, 64],
    dropout=0.3,
).to(DEVICE)

print("Calculando class imbalance SOLO con train...")
n_neg, n_pos = compute_class_balance(
    paths=PATHS,
    target_col=TARGET_COL,
    batch_size_rows=BATCH_SIZE_ROWS,
    region=AWS_REGION,
)

print(f"Negativos: {n_neg}")
print(f"Positivos: {n_pos}")

if n_pos == 0:
    raise ValueError("No hay ejemplos positivos en train.")

pos_weight_value = n_neg / max(n_pos, 1)
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

print("Entrenando...")
for epoch in range(1, EPOCHS + 1):
    train_metrics = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=DEVICE,
    )
    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_metrics['loss']:.6f} | "
        f"train_acc={train_metrics['accuracy']:.6f}"
    )

In [ ]:
# Guardamos el modelo
torch.save(
        {
            "model_state_dict": model.state_dict(),
            "numeric_cols": NUMERIC_COLS,
            "target_col": TARGET_COL,
            "id_col": ID_COL,
            "hash_bucket_size": HASH_BUCKET_SIZE,
            "embedding_dim": EMBEDDING_DIM,
            "scaler_mean": scaler_mean,
            "scaler_std": scaler_std,
        },
        "tabular_model_artifacts.pt",
    )

# 5.2 Evaluation

In [ ]:
def compute_binary_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """
    Calcula precision, recall, accuracy y f1-score para clasificación binaria.
    """
    y_true = y_true.astype(np.int64)
    y_pred = y_pred.astype(np.int64)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)

    return {
        "precision": float(precision),
        "recall": float(recall),
        "accuracy": float(accuracy),
        "f1_score": float(f1),
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
    }

In [ ]:
@torch.no_grad()
def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
    device: str,
    threshold: float = 0.5,
) -> dict[str, float]:
    """
    Evalúa el modelo sobre un loader y devuelve:
    precision, recall, accuracy y f1-score.
    """
    model.eval()

    all_y_true = []
    all_y_prob = []

    for batch in loader:
        x_num = batch["x_num"].to(device, non_blocking=True)
        x_id = batch["x_id"].to(device, non_blocking=True)
        y = batch["y"].to(device, non_blocking=True)

        logits = model(x_num, x_id)
        probs = torch.sigmoid(logits)

        all_y_true.append(y.cpu().numpy())
        all_y_prob.append(probs.cpu().numpy())

    y_true = np.concatenate(all_y_true).astype(np.int64)
    y_prob = np.concatenate(all_y_prob)
    y_pred = (y_prob >= threshold).astype(np.int64)

    metrics = compute_binary_metrics(y_true, y_pred)
    return metrics

Para evaluar, usaremos las particiones descritas en `VALID_PATHS`:

In [ ]:
print("Construyendo dataset de validación...")
valid_ds = S3ParquetArrowIterableDataset(
    paths=VALID_PATHS,
    target_col=TARGET_COL,
    numeric_cols=NUMERIC_COLS,
    id_col=ID_COL,
    batch_size_rows=BATCH_SIZE_ROWS,
    hash_bucket_size=HASH_BUCKET_SIZE,
    fill_null_value=FILL_NULL_VALUE,
    scaler_mean=scaler_mean,
    scaler_std=scaler_std,
    shuffle_within_batch=False,
    region=AWS_REGION,
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=TRAIN_BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_tabular,
)

In [ ]:
final_valid_metrics = evaluate_model(
    model=model,
    loader=valid_loader,
    device=DEVICE,
    threshold=0.5,
)

print("Valid metrics:")
for k, v in final_valid_metrics.items():
    print(f"{k}: {v}")

# 6. Resultados

Entrenado solo con la última partición `cutoff_date=2025-08-01` presente en `s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_entrena_target_independiente` y validando con todos los datos de `s3://ada-us-east-1-sbx-live-mx-czl2-data/analytical/myedv/abt_valida_target_independiente`, se obtuvieron los siguientes resultados:

| Métrica   | Random Forest (M1) | Neural Network (M2) | Diferencia (M2 - M1) | Variación (%) |
|-----------|-------------------|--------------------|----------------------|---------------|
| Accuracy  | 0.7452            | 0.7498             | +0.0046              | +0.62%        |
| Precision | 0.7059            | 0.7389             | +0.0329              | +4.66%        |
| Recall    | 0.6704            | 0.6972             | +0.0268              | +4.00%        |
| F1-score  | 0.6877            | 0.7174             | +0.0297              | +4.32%        |